In [ ]:
library(tidyverse)
library(bigrquery)

# The below queries represent dataset AllofUsControlledTierDatasetv8 in cohort cohort_tsh at 20260509_032326.

# If EXPORT_BUCKET is not set in your environment, uncomment and update the line below
# with your actual Cloud Storage bucket name from the Resources tab in your workspace.
Sys.setenv(EXPORT_BUCKET = "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055")
BILLING = "wb-silky-pepper-6055"

In [ ]:
# Domain person
person_sql = r"(

    SELECT
        date_of_birth,
        ethnicity,
        T_DISP_ethnicity,
        ethnicity_concept_id,
        gender,
        T_DISP_gender,
        gender_concept_id,
        person_id,
        race,
        T_DISP_race,
        race_concept_id,
        self_reported_category,
        T_DISP_self_reported_category,
        self_reported_category_concept_id,
        sex_at_birth,
        T_DISP_sex_at_birth,
        sex_at_birth_concept_id 
    FROM
        `wb-silky-artichoke-2408.C2024Q3R8_index_111825`.T_ENT_person 
    WHERE
        (
            has_whole_genome_variant = true
        ) 
        AND (
            has_ehr_data = true
        ) 
        AND (
            id IN (SELECT
                person_id AS primary_id 
            FROM
                `wb-silky-artichoke-2408.C2024Q3R8_index_111825`.T_ENT_measurementOccurrence 
            WHERE
                measurement_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2024Q3R8_index_111825`.T_HAD_measurementLoincConcept_default 
                WHERE
                    ancestor = 37062351 
                UNION
                ALL SELECT
                    37062351))
            ))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "person",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person",
  "person_*.csv")
message(str_glue('The data will be written to {person_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2024Q3R8_index_111825", person_sql, billing = BILLING),
  person_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

person_df <- read_bq_export_from_workspace_bucket(person_path)
dim(person_df)
head(person_df, 5)


In [ ]:
# Domain measurementOccurrence
measurementOccurrence_sql = r"(

    SELECT
        measurement_concept_id,
        measurement_datetime,
        measurement_source_concept_id,
        measurement_source_value,
        measurement_type_concept_id,
        measurement_type_concept_name,
        T_DISP_measurement_type_concept_name,
        operator_concept_id,
        operator_concept_name,
        T_DISP_operator_concept_name,
        person_id,
        range_high,
        range_low,
        source_concept_code,
        T_DISP_source_concept_code,
        source_concept_name,
        T_DISP_source_concept_name,
        source_vocabulary,
        T_DISP_source_vocabulary,
        standard_concept_code,
        T_DISP_standard_concept_code,
        standard_concept_name,
        T_DISP_standard_concept_name,
        standard_vocabulary,
        T_DISP_standard_vocabulary,
        unit_concept_id,
        unit_concept_name,
        T_DISP_unit_concept_name,
        unit_source_value,
        value_as_concept_id,
        value_as_concept_name,
        T_DISP_value_as_concept_name,
        value_as_number,
        value_source_value,
        visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2024Q3R8_index_111825`.T_ESA_measurementOccurrence_person_id 
    WHERE
        (
            person_id IN (SELECT
                id 
            FROM
                `wb-silky-artichoke-2408.C2024Q3R8_index_111825`.T_ENT_person 
            WHERE
                (has_whole_genome_variant = true) 
                AND (has_ehr_data = true) 
                AND (id IN (SELECT
                    person_id AS primary_id 
                FROM
                    `wb-silky-artichoke-2408.C2024Q3R8_index_111825`.T_ENT_measurementOccurrence 
                WHERE
                    measurement_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2024Q3R8_index_111825`.T_HAD_measurementLoincConcept_default 
                    WHERE
                        ancestor = 37062351 
                    UNION
                    ALL SELECT
                        37062351))))
                ) 
                AND (
                    measurement_concept_id IN (SELECT
                        descendant 
                    FROM
                        `wb-silky-artichoke-2408.C2024Q3R8_index_111825`.T_HAD_measurementLoincConcept_default 
                    WHERE
                        ancestor = 37062351 
                    UNION
                    ALL SELECT
                        37062351)
                ))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
measurementOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports",
  "measurementOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "measurementOccurrence",
  "measurementOccurrence_*.csv")
message(str_glue('The data will be written to {measurementOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2024Q3R8_index_111825", measurementOccurrence_sql, billing = BILLING),
  measurementOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {measurementOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

measurementOccurrence_df <- read_bq_export_from_workspace_bucket(measurementOccurrence_path)
dim(measurementOccurrence_df)
head(measurementOccurrence_df, 5)
